# 共通テスト実行ノートブック

ソースノートブックを実行し、テスト結果を HTML レポートで表示する。

**ウィジェット:**
| パラメータ | 説明 |
|---|---|
| `P_BASE_YM` | 基準年月（例: 200512） |
| `P_SOURCE_NOTEBOOK` | ソースコードへのパス（相対パス可） |

**テストファイル命名規約:**

ソースノートブック名からハイフンをアンダースコアに置換し `test_` プレフィックスを付与。

例: `import-source-files` → `test_import_source_files.py`

In [0]:
%pip install pytest pytest-html -q

In [0]:
import sys
sys.dont_write_bytecode = True

dbutils.widgets.text("P_BASE_YM", defaultValue="200512", label="基準年月")
dbutils.widgets.text(
    "P_SOURCE_NOTEBOOK",
    defaultValue="../src/",
    label="ソースコードのパス",
)

In [0]:
import os

base_ym = dbutils.widgets.get("P_BASE_YM")
source_notebook = dbutils.widgets.get("P_SOURCE_NOTEBOOK")
source_name = os.path.basename(source_notebook)

print(f"{source_name} を実行中... (P_BASE_YM={base_ym})")
dbutils.notebook.run(source_notebook, timeout_seconds=600, arguments={"P_BASE_YM": base_ym})
print(f"{source_name} 完了")

In [0]:
import os
import sys
import tempfile
import pytest

sys.dont_write_bytecode = True

base_ym = dbutils.widgets.get("P_BASE_YM")
source_notebook = dbutils.widgets.get("P_SOURCE_NOTEBOOK")

# ── ソース名からテストファイルを自動導出 ──
project_root = ".."
source_name = os.path.basename(source_notebook)
test_file = os.path.join(project_root, "tests", f"test_{source_name.replace('-', '_')}.py")
print(f"テストファイル: {test_file}")

# ── 環境変数設定 ──
os.environ["P_BASE_YM"] = base_ym
os.environ["SOURCE_NOTEBOOK"] = source_notebook

# ── テストモジュールのキャッシュクリア（ファイル更新後の再実行に対応） ──
test_module_name = os.path.splitext(os.path.basename(test_file))[0]
for mod_name in list(sys.modules.keys()):
    if test_module_name in mod_name:
        del sys.modules[mod_name]

# ── pytest 実行 ──
html_path = os.path.join(tempfile.mkdtemp(), "report.html")

exit_code = pytest.main([
    test_file,
    "-v",
    "--tb=short",
    "--no-header",
    "-p", "no:cacheprovider",
    "--import-mode=importlib",
    f"--html={html_path}",
    "--self-contained-html",
])

# ── HTML レポートをノートブック上に表示 ──
with open(html_path, "r") as f:
    displayHTML(f.read())

# ── テスト結果の判定 ──
if exit_code != 0:
    raise Exception(f"テスト失敗 (exit code: {exit_code})")